# 02. Framingham Data Cleaning & Preprocessing Pipeline
**RoboDoctor AI — Phase 3: Reproducible Clinical Data Cleaning**
*Author: Senior Machine Learning Engineer & Healthcare AI Engineer*

---

## 1. Objectives & Clinical Rationale
In healthcare AI, naive cleaning (such as dropping rows or replacing NaN with zero) introduces severe selection bias and clinical distortion. This notebook establishes a reproducible, leakage-free data cleaning pipeline adhering to:
1. **Zero arbitrary row dropping:** All 4,240 records are preserved.
2. **Physiological BP correction:** Rectifying the 82 cases where systolic BP was recorded below diastolic BP (`sysBP <= diaBP`).
3. **Logically consistent smoking imputation:** If non-smoker (`currentSmoker == 0`), missing cigarettes are assigned 0.0. If smoker (`currentSmoker == 1`), missing cigarettes receive the smoker-specific median.
4. **Pipeline Encapsulation:** Creating a `ClinicalDataCleaner` class adhering to scikit-learn's `TransformerMixin` so all statistics are learned strictly from training folds.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer

csv_path = os.path.join('..', 'data', 'framingham.csv')
df = pd.read_csv(csv_path)
print(f"Loaded raw dataset: {df.shape}")


Loaded raw dataset: (4240, 16)


## 2. Inverted Blood Pressure Correction
Systolic pressure reflects peak ventricular contraction and must physiologically exceed diastolic resting vascular resistance. The 82 records where `sysBP <= diaBP` represent recording inversion.


In [2]:
inverted_before = (df['sysBP'] <= df['diaBP']).sum()
print(f"Inverted BP cases before correction: {inverted_before}")

def correct_bp_inversion(df_in):
    df_out = df_in.copy()
    sys_corrected = np.maximum(df_out['sysBP'], df_out['diaBP'])
    dia_corrected = np.minimum(df_out['sysBP'], df_out['diaBP'])
    df_out['sysBP'] = sys_corrected
    df_out['diaBP'] = dia_corrected
    return df_out

df_bp_fixed = correct_bp_inversion(df)
inverted_after = (df_bp_fixed['sysBP'] <= df_bp_fixed['diaBP']).sum()
print(f"Inverted BP cases after correction: {inverted_after}")
print(f"Minimum Pulse Pressure after correction: {(df_bp_fixed['sysBP'] - df_bp_fixed['diaBP']).min():.2f} mmHg")


Inverted BP cases before correction: 82
Inverted BP cases after correction: 0
Minimum Pulse Pressure after correction: 0.10 mmHg


## 3. Context-Aware Smoking Imputation
- When `currentSmoker == 0`, missing `cigsPerDay` is logically 0 (16 cases).
- When `currentSmoker == 1`, missing `cigsPerDay` reflects active smoker intensity (19 cases), imputed using the median among active smokers.


In [3]:
smoker_median_cigs = df[df['currentSmoker'] == 1]['cigsPerDay'].median()
print(f"Median cigarettes/day for active smokers: {smoker_median_cigs}")

def impute_smoking_consistency(df_in, smoker_median=15.0):
    df_out = df_in.copy()
    # Non-smokers with NaN cigs -> 0.0
    df_out.loc[(df_out['currentSmoker'] == 0) & (df_out['cigsPerDay'].isna()), 'cigsPerDay'] = 0.0
    # Active smokers with NaN cigs -> smoker median
    df_out.loc[(df_out['currentSmoker'] == 1) & (df_out['cigsPerDay'].isna()), 'cigsPerDay'] = smoker_median
    return df_out

df_smoke_fixed = impute_smoking_consistency(df_bp_fixed, smoker_median_cigs)
print(f"Missing cigsPerDay remaining: {df_smoke_fixed['cigsPerDay'].isna().sum()}")


Median cigarettes/day for active smokers: 15.0
Missing cigsPerDay remaining: 0


## 4. Scikit-Learn Leakage-Free Clinical Transformer
We now construct the full `ClinicalDataCleaner` class implementing `fit` and `transform`.
This ensures that `smoker_median_cigs`, categorical modes, and numeric medians are **strictly learned on training data** and never leak from validation or test splits.


In [4]:
class ClinicalDataCleaner(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.smoker_median_cigs_ = 15.0
        self.numeric_medians_ = {}
        self.categorical_modes_ = {}
        self.numeric_cols = ['totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']
        self.cat_cols = ['BPMeds', 'education']

    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # Calculate smoker median
        active_smokers = X_df[X_df['currentSmoker'] == 1]['cigsPerDay']
        self.smoker_median_cigs_ = float(active_smokers.median()) if not active_smokers.empty else 15.0
        
        # Calculate numeric medians
        for col in self.numeric_cols:
            if col in X_df.columns:
                self.numeric_medians_[col] = float(X_df[col].median())
                
        # Calculate categorical modes
        for col in self.cat_cols:
            if col in X_df.columns:
                mode_vals = X_df[col].mode()
                self.categorical_modes_[col] = float(mode_vals.iloc[0]) if not mode_vals.empty else 0.0
                
        return self

    def transform(self, X):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # 1. Correct BP inversion
        if 'sysBP' in X_df.columns and 'diaBP' in X_df.columns:
            sys_corr = np.maximum(X_df['sysBP'], X_df['diaBP'])
            dia_corr = np.minimum(X_df['sysBP'], X_df['diaBP'])
            X_df['sysBP'] = sys_corr
            X_df['diaBP'] = dia_corr
            
        # 2. Impute smoking consistency
        if 'currentSmoker' in X_df.columns and 'cigsPerDay' in X_df.columns:
            X_df.loc[(X_df['currentSmoker'] == 0) & (X_df['cigsPerDay'].isna()), 'cigsPerDay'] = 0.0
            X_df.loc[(X_df['currentSmoker'] == 1) & (X_df['cigsPerDay'].isna()), 'cigsPerDay'] = self.smoker_median_cigs_
            
        # 3. Impute categorical modes
        for col, mode_val in self.categorical_modes_.items():
            if col in X_df.columns:
                X_df[col] = X_df[col].fillna(mode_val)
                
        # 4. Impute numeric medians
        for col, med_val in self.numeric_medians_.items():
            if col in X_df.columns:
                X_df[col] = X_df[col].fillna(med_val)
                
        return X_df

# Test the transformer
cleaner = ClinicalDataCleaner()
cleaner.fit(df.drop(columns=['TenYearCHD']))
cleaned_df = cleaner.transform(df.drop(columns=['TenYearCHD']))

print("Missing values after ClinicalDataCleaner:")
print(cleaned_df.isnull().sum())
print(f"\nAll 4240 records cleaned with 0 NaNs: {cleaned_df.isnull().sum().sum() == 0}")


Missing values after ClinicalDataCleaner:
male               0
age                0
education          0
currentSmoker      0
cigsPerDay         0
BPMeds             0
prevalentStroke    0
prevalentHyp       0
diabetes           0
totChol            0
sysBP              0
diaBP              0
BMI                0
heartRate          0
glucose            0
dtype: int64

All 4240 records cleaned with 0 NaNs: True


## 5. Summary of Phase 3 Cleaning Results
- Inverted blood pressures corrected physically ($PP \ge 0$).
- Cigarettes/day logically stratified by smoking status.
- Fully compatible with `sklearn.pipeline.Pipeline` for zero-leakage cross-validation.
